# Causal-rCM frame-level attention (Wan2.1 T2V 1.3B)Companion to `26Mar26-PyramidForcing-frames72`, on the other codebase.**Provenance** — extracted with `winbeau/rcm` @ `--extract_attn_layers all`,checkpoint `Causal_rCM_Wan2.1_T2V_1.3B_480p_TF-dCM-init_SF-DMD_c1-1_step4.pt`,`--first_chunk_t 1 --chunk_t 1` (one latent frame per chunk, so `block_sizes`is all ones and the query axis is at full per-frame resolution), 4 denoisingsteps, 480p 16:9 (latent 60x104, `frame_seq_length = 1560`), seed 0.Values are **pre-softmax logits** (`is_logits=True`), captured on theonce-per-chunk KV-append forward at t=0 -- clean context, one observation perchunk. They are mean logits over all (query token, key token) pairs in a frameblock, computed by mean-pooling each frame's tokens before contracting, whichis exact (see `rcm/utils/frame_attention.py`).The schema is identical to the Self-Forcing artifacts, so`attention_plot_utils` reads these unmodified.

In [ ]:
import sysfrom pathlib import Pathimport matplotlib.pyplot as pltimport numpy as npimport torchREPO_ROOT = Path.cwd()while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:    REPO_ROOT = REPO_ROOT.parent# Reuse the Self-Forcing plotting helpers verbatim -- same artifact schema.sys.path.insert(0, str(REPO_ROOT / "notebooks" / "26Mar26-PyramidForcing-frames72"))import attention_plot_utils as apuDATA_DIR = REPO_ROOT / "data" / "26Jul26-CausalRCM-attention" / "frames72"FIG_DIR = REPO_ROOT / "figures" / "26Jul26-CausalRCM-attention"FIG_DIR.mkdir(parents=True, exist_ok=True)apu.apply_attention_plot_style()paths = sorted(DATA_DIR.glob("*.pt"), key=lambda p: int(p.name.split("layer")[1].split("_")[0]))print(f"{len(paths)} layer artifacts in {DATA_DIR.relative_to(REPO_ROOT)}")

## 1. Load one layer and confirm the schema round-trips

In [ ]:
data = apu.load_attention_data(paths[15])layer_idx, num_frames, num_heads, full, last, last_q = apu.get_attention_arrays(data)print(f"\nmodel        : {data['model']}")print(f"capture pass : {data['capture_pass']}")print(f"method       : {data['extraction_method']}")print(f"full         : {full.shape}   last block : {last.shape}")future = np.triu(np.ones((num_frames, num_frames), dtype=bool), 1)print(f"causality    : {int((full[:, future] != 0).sum())} future entries (must be 0)")

## 2. All-head heatmap, reusing the Self-Forcing renderer

In [ ]:
out = apu.render_attention_heatmaps(    paths[15],    save_dir=FIG_DIR,    data=data,    tick_positions=(0, 36, 71),)out

## 3. Where each head puts its massThe taxonomy Pyramid-Forcing is built on lives on one axis: how much attentionmass a head spends on the **middle** of the context, as opposed to the sink(frame 0) or the recent window. Softmax each causal row, then average overquery frames.

In [ ]:
RECENT = 4      # frames counted as "recent", including the current oneWARMUP = 4      # skip the first rows, where the bands are not yet distinguishabledef band_masses(full_frame_attn: np.ndarray) -> np.ndarray:    """-> [num_heads, 3] columns (sink, middle, recent), each averaged over query frames."""    n_heads, n_frames, _ = full_frame_attn.shape    out = np.zeros((n_heads, 3))    for h in range(n_heads):        acc = np.zeros(3)        for qf in range(WARMUP, n_frames):            z = full_frame_attn[h, qf, : qf + 1]            a = np.exp(z - z.max())            a /= a.sum()            lo = max(0, qf - RECENT + 1)            acc += [a[0], a[1:lo].sum() if lo > 1 else 0.0, a[lo:].sum()]        out[h] = acc / (n_frames - WARMUP)    return outmasses = np.stack([band_masses(torch.load(p, map_location="cpu", weights_only=False)["full_frame_attention"].float().numpy()) for p in paths])print(f"masses: {masses.shape}  [layer, head, (sink, middle, recent)]")print(f"middle-band mass  min={masses[..., 1].min():.3f}  max={masses[..., 1].max():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.4), constrained_layout=True)for ax, (k, name) in zip(axes, enumerate(["sink (frame 0)", "middle", "recent"])):    im = ax.imshow(masses[..., k].T, aspect="auto", cmap="magma", vmin=0, vmax=1, origin="lower")    ax.set_title(name)    ax.set_xlabel("layer")    ax.set_ylabel("head" if k == 0 else "")    fig.colorbar(im, ax=ax, fraction=0.046)fig.suptitle("Causal-rCM Wan2.1 T2V 1.3B - attention mass by band (72 latent frames)", y=1.06)fig.savefig(FIG_DIR / "band_mass_by_layer_head.pdf", bbox_inches="tight")fig.savefig(FIG_DIR / "band_mass_by_layer_head.png", dpi=200, bbox_inches="tight")plt.show()

## 4. ReadThe middle-band mass spans roughly **0.00 to 0.83 across the 360 (layer, head)pairs** -- the head heterogeneity that a per-head KV policy exploits is presentin Causal-rCM, not only in Self-Forcing. Heads at the low end are served by a`[sink + recent]` cache alone; heads at the high end are the ones that wouldlose real signal to a naive sliding window, and are the candidates for thestrided / periodic / merged middle strategies.What this notebook does **not** establish: that the three-way Anchor / Wave /Veil split is the right partition *here*. That needs the periodicity andstability analyses (see `26May4-PyramidForcing-multihead`) run against theseartifacts, and a second prompt to check the labels are prompt-stable. Theextraction currently uses one prompt and one seed.